# Stage 9b, observe Spark: what the Spark History MCP server can tell you

Stage 9 asked an agent about the pipeline. Here the focus is Spark itself. The notebook runs a small workload on the
lab's Spark Connect server, reads its metrics with the **Spark History MCP** tools **by hand**, then lets an agent (built
on [AWS Strands](https://strandsagents.com), `lab_agents.py`) diagnose it.

The one MCP endpoint serves two backends: `server="live"` is the running Spark Connect session (the queries you just
ran), `server="history"` is the History Server (finished applications). On S3 a running application's event log only
reaches the History Server when the application stops, so the History Server cannot list the current session: that is
why the live backend exists. The server is **read-only**.

The workload runs three queries: a full scan, the same scan with a partition filter, and a join with one very hot key.
Without an LLM (`llm.provider=none`) everything up to the agent runs; the agent cell prints `skipped (no LLM configured)`.

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import lab_data
import lab_utils as lu

lu.require_ops()
if "spark_history" not in lu.ops_servers():
    raise RuntimeError("The Spark History MCP server is off: set sparkHistoryMcp.enabled=true (and sparkHistory.enabled).")
llm_ready = lu.llm_available()
if llm_ready:
    import lab_agents as la
    lu.require_tool_model()
print("tools:", ", ".join(sorted(t["name"] for t in lu.ops_tools("spark_history"))))

def short(obj, n=1200):
    text = obj if isinstance(obj, str) else json.dumps(obj, indent=1, default=str)
    print(text if len(text) <= n else text[:n] + f"\n... ({len(text) - n} more characters)")

## 1. A workload with something to find
`lab_data.order_events()` replicated 200 times into an Iceberg table partitioned by `status`; a scan over all of it, the
same scan restricted to one partition, and a sort-merge join (broadcast and adaptive skew handling off) where 90% of the
rows share the key `0`. Names are fixed, so the notebook can run again.

In [ ]:
from pyspark.sql import functions as F

spark = lu.get_spark("ops02-workload")
table = f"{lu.CATALOG}.shop_bronze.ops02_events"
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {lu.CATALOG}.shop_bronze")
spark.sql(f"DROP TABLE IF EXISTS {table}")
rows = [{k: e[k] for k in ("order_id", "customer_id", "product_id", "quantity", "order_ts", "status")} for e in lab_data.order_events()]
schema = "order_id long, customer_id long, product_id long, quantity int, order_ts string, status string"
(spark.createDataFrame(rows, schema).crossJoin(spark.range(200).withColumnRenamed("id", "copy"))
      .writeTo(table).partitionedBy(F.col("status")).using("iceberg").create())

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
full = spark.table(table).agg(F.count("*").alias("rows"), F.sum("quantity").alias("units")).collect()[0]
filtered = spark.table(table).where("status = 'returned'").agg(F.count("*").alias("rows"), F.sum("quantity").alias("units")).collect()[0]
left = spark.range(2_000_000).select(F.when(F.col("id") % 10 < 9, F.lit(0)).otherwise(F.col("id")).alias("k"))
skewed = left.join(spark.range(1000).withColumnRenamed("id", "k"), "k").count()
print("full scan:", full.asDict(), "\nfiltered :", filtered.asDict(), "\nskewed join rows:", skewed)

## 2. The tools by hand, on the live session
Every MCP tool is a function with a JSON schema. First the applications the **live** backend knows (one: the Spark
Connect server), then its SQL executions, newest queries last. The ids change when Spark Connect restarts, so the
notebook never hardcodes one.

In [ ]:
def call(name, **args):
    return lu.ops_call("spark_history", name, {"server": "live", **args})

def items(res):
    """The list inside a tool result (some tools return a bare list, others wrap it)."""
    if isinstance(res, list):
        return res
    if isinstance(res, dict):
        for v in res.values():
            if isinstance(v, list):
                return v
    return []

apps = items(call("list_applications"))
short(apps)
app_id = (apps[0].get("id") or apps[0].get("app_id")) if apps and isinstance(apps[0], dict) else None
print("live application:", app_id)

In [ ]:
sqls = items(call("list_sql_executions", app_id=app_id)) if app_id else []
for q in sqls[-6:]:
    print({k: q.get(k) for k in ("id", "status", "duration", "description") if isinstance(q, dict)})

## 3. Where did the time go?
`get_job_bottlenecks` ranks the slowest stages and tasks of the application and flags skew (the longest task against
the median). The hot-key join above should be at the top. `compare_sql_executions` puts two queries side by side:
the full scan against the one with the partition filter, which should read a fraction of the data.

In [ ]:
if app_id:
    short(call("get_job_bottlenecks", app_id=app_id), 2500)

In [ ]:
ids = [q["id"] for q in sqls if isinstance(q, dict) and "id" in q]
if app_id and len(ids) >= 2:
    short(call("compare_sql_executions", app_id=app_id, execution_id1=ids[-3] if len(ids) >= 3 else ids[0], execution_id2=ids[-2]), 2500)
else:
    print("fewer than two SQL executions on the live backend yet; re-run the workload cell")

## 4. Finished applications, on the History Server
Completed applications (the earlier notebooks' Spark Connect sessions, the SparkApplications of the e2e) live on the
`history` backend. A fresh lab, or one older than the 7-day cleaner, has none: that is fine.

In [ ]:
try:
    done = items(lu.ops_call("spark_history", "list_applications", {"server": "history"}))
except Exception as e:
    done = []
    print("history backend not reachable:", type(e).__name__, e)
print(f"{len(done)} finished application(s) in the History Server" + ("" if done else
      " (a running application's events reach S3 only when it stops, so the current session is not listed)"))
short(done[:3], 1000)

## 5. Ask the agent
`lab_agents.ops_agent(("spark_history",))` offers the read-only Spark tools only. Its prompt says which server holds
what (`live` for the running session, `history` for finished applications). Every tool call is printed under the
answer, so you can check that the answer comes from the tools.

In [ ]:
import pandas as pd

QUESTIONS = ["Why was my last query slow?",
             "Which stage had the most skew?",
             "How much less data did the filtered query read than the full scan?"]
if lu.skip_unless_llm("the agent diagnoses the workload"):
    with la.ops_agent(("spark_history",)) as agent:
        for q in QUESTIONS:
            out = la.ask(agent, q)
            print(f"Q: {q}\n\nA: {out['answer']}\n")
            display(pd.DataFrame([{"tool": s["tool"], "args": json.dumps(s["args"]),
                                   "result": json.dumps(s.get("result"), default=str)[:200]} for s in out["steps"]]))

## What to look for
* The skewed join's stage has one task far longer than the median: `get_job_bottlenecks` shows it and the agent
  should name that stage. The numbers are in the tool results, not in the model's imagination.
* The filtered scan reads one partition of the table: compare the input bytes or rows of the two executions.
* The server is **read-only**: the agent can look, not kill a job or change a setting.
* Small models sometimes pick the wrong backend (`history` for the live session) or skip a step; the steps table is how
  you check. A bigger tool model (`llm.toolModel`) does this more reliably.